##关于本文件的说明：  
这是Action-triggered-RAG的原型代码，源自Genearative AI and AI Applications课程的个人作业，是本人原创的RAG系统。  

与沙盘相似，这份原型代码聚焦于欧洲碳交易市场下的相关公司的定价和ROE，以及与之配套的动作触发式RAG。与沙盘不同的是，这份原型代码更注重RAG的搭建而非对于现实经济活动的模拟。所以，在这份代码里，你将看到更加全面且复杂的RAG搭建过程。具体过程可见下一部分英文概览以及每一步的简介。  

原型代码用于验证动作触发式RAG的可行性，和了解它的搭建过程。在此基础上，我才得以建构更加真实的沙盘决策模拟器。

##A Decision-Triggered RAG System for EU ETS Carbon Market Analysis

###Overview
Step 1 — Environment Setup  
Step 2 — PDF Ingestion, Chunking, Embedding, Store to ChromaDB  
Step 3 — Query Rewriting: using LLM Translates Sliders' State into Retrieval Query  
Step 4 — Retrieval with Metadata Filtering  
Step 5 — Simulation Engine + ROE Calculator  
Step 6 — Prompt Assembly  
Step 7 — Baseline Pipeline  
Step 8 — ipywidgets Dashboard + Report Export (.docx)  
Step 9 — Evaluation

###Step1:Environment Setup  
**Objective:**
Install all required packages and confirm the environment is ready before writing any pipeline code.  


In [ ]:
!pip install pymupdf chromadb langchain google-generativeai python-docx ipywidgets sentence-transformers langchain_text_splitters rank-bm25 anthropic -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [ ]:
import chromadb
import langchain
import google.generativeai as genai
print("All good")

All good


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('ANTHROPIC_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)
print("✓ API configured.")

✓ API configured.


In [ ]:
import anthropic
import time
_client = anthropic.Anthropic(api_key="your key")

def llm_call(prompt: str, retries: int = 5) -> str:
    for attempt in range(retries):
        try:
            message = _client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=4096,
                messages=[{"role": "user", "content": prompt}]
            )
            time.sleep(3)
            return message.content[0].text.strip()
        except Exception as e:
            if "429" in str(e) or "overloaded" in str(e).lower():
                wait = 30
                print(f"Rate limit hit. Waiting {wait}s...")
                time.sleep(wait)
            else:
                raise e
    raise RuntimeError("Max retries exceeded.")

###Step2:PDF Ingestion, Metadata, Chunking, Embedding & ChromaDB Storage  
**Objective:**
Ingesting reference files `2025 IACP Report.pdf`, `2026 ICAP Report.pdf` and `2025 Carbon Market Report.pdf` as the knowledge base for RAG. Metadata ensures each chunk with source, year, region, and page. Using keywords and `BM25` to filter chunks, and after chunking, embeddings run locally via `sentence-transformers`. The outputs were stored in cloud-based drive, so that when running this code next time, embedded chunks can be reloaded from the drive rather than rerun the whole step2.

In [ ]:
import pymupdf
import chromadb
import shutil, re, os
import numpy as np
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import files as colab_files
from rank_bm25 import BM25Okapi
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# ── Config ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
CHROMA_PATH = "/content/drive/MyDrive/ets_rag/chromadb"
DATA_DIR    = Path("/content/ets_pdfs")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Upload PDFs ──────────────────────────────────────────
print("Select all PDF files to upload...")
uploaded = colab_files.upload()

PDF_FILES = []

for fname in uploaded.keys():
    clean = re.sub(r"[^A-Za-z0-9._-]+", "_", fname).strip("_")
    dest = DATA_DIR / clean
    shutil.move(fname, dest)
    filename_lower = clean.lower()

    if "icap" in filename_lower:
        source = "ICAP"
    elif "carbon" in filename_lower:
        source = "CarbonMarket"
    elif "ets" in filename_lower:
        source = "ETS"
    else:
        source = "Unknown"

    year_match = re.search(r"(20\d{2})", filename_lower)
    year = year_match.group(1) if year_match else "unknown"

    if "china" in filename_lower:
        region = "China"
    elif "uk" in filename_lower:
        region = "UK"
    elif "usa" in filename_lower:
        region = "USA"
    else:
        region = "EU"

    PDF_FILES.append({
        "path": str(dest),
        "filename": Path(clean).stem,
        "source": source,
        "year": year,
        "region": region
    })
    print(f"✓ Saved: {dest} | source={source}, year={year}, region={region}")

# ── Extract EU-relevant pages ────────────────────────────
EU_SECTION_KEYWORDS = [
    "european union", "eu ets", "eua", "europe",
    "market stability reserve", "msr", "auction",
    "allowance", "carbon price", "emission trading",
    "cbam", "carbon border"
]

def is_eu_relevant_page(text):
    return any(kw in text.lower() for kw in EU_SECTION_KEYWORDS)

def extract_eu_pages(pdf_config):
    doc = pymupdf.open(pdf_config["path"])
    pages, skipped = [], 0
    for page_num, page in enumerate(doc):
        text = page.get_text().strip()
        if len(text) < 100:
            skipped += 1
            continue
        if pdf_config["source"] == "ICAP" and not is_eu_relevant_page(text):
            skipped += 1
            continue
        pages.append({
            "text": text,
            "page": page_num + 1,
            "source": pdf_config["source"],
            "year": pdf_config["year"],
            "region": pdf_config["region"]
        })
    print(f"{pdf_config['source']} ({pdf_config['year']}): "
          f"kept {len(pages)} pages, skipped {skipped}")
    return pages

all_pages = []
for pdf in PDF_FILES:
    all_pages.extend(extract_eu_pages(pdf))

print(f"\nTotal EU-relevant pages: {len(all_pages)}")

# ── Chunk ────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
)

all_chunks = []
for page in all_pages:
    splits = splitter.split_text(page["text"])
    for i, split in enumerate(splits):
        all_chunks.append({
            "text": split,
            "metadata": {
                "source": page["source"],
                "year": page["year"],
                "region": page["region"],
                "page": page["page"],
                "chunk_index": i
            }
        })

print(f"Total chunks after chunking: {len(all_chunks)}")

# ── Keyword filter ───────────────────────────────────────
RELEVANT_KEYWORDS = [
    "carbon price", "eua", "allowance price", "ets price",
    "market stability reserve", "msr", "auction", "cap",
    "supply", "demand", "emission reduction", "ets reform",
    "free allocation", "cbam", "carbon leakage", "revenue"
]

def is_relevant(text):
    return any(kw in text.lower() for kw in RELEVANT_KEYWORDS)

relevant_chunks = [c for c in all_chunks if is_relevant(c["text"])]
print(f"Chunks after keyword filter: {len(relevant_chunks)}")

# ── BM25 filter → keep top 400 ───────────────────────────
tokenised    = [c["text"].lower().split() for c in relevant_chunks]
bm25         = BM25Okapi(tokenised)
query_tokens = "EU ETS carbon price auction MSR allowance supply demand".split()
scores       = bm25.get_scores(query_tokens)
top_indices  = np.argsort(scores)[::-1][:400]
final_chunks = [relevant_chunks[i] for i in sorted(top_indices)]

print(f"Chunks after BM25 filter:   {len(final_chunks)}")

# ── Embed locally + store in ChromaDB ───────────────────
# all-MiniLM-L6-v2: fast, lightweight, no API cost
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Drop existing collection if rebuilding from scratch
try:
    client.delete_collection("ets_knowledge_base")
    print("Cleared existing collection")
except:
    pass

collection = client.get_or_create_collection(
    name="ets_knowledge_base",
    embedding_function=embedding_fn
)

BATCH_SIZE = 50

for i in range(0, len(final_chunks), BATCH_SIZE):
    batch     = final_chunks[i:i + BATCH_SIZE]
    texts     = [c["text"] for c in batch]
    metadatas = [c["metadata"] for c in batch]
    ids       = [f"chunk_{i + j}" for j in range(len(batch))]

    collection.add(
        documents=texts,   # ChromaDB calls embedding_fn automatically
        metadatas=metadatas,
        ids=ids
    )
    print(f"Stored batch {i // BATCH_SIZE + 1} / "
          f"{-(-len(final_chunks) // BATCH_SIZE)}")

print(f"\nDone. Total chunks in ChromaDB: {collection.count()}")

# ── Sanity check ─────────────────────────────────────────
results = collection.query(
    query_texts=["What drove EU carbon prices in 2024?"],
    n_results=3,
    where={"region": "EU"}
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {results['metadatas'][0][i]['source']}, "
          f"Page: {results['metadatas'][0][i]['page']}")
    print(doc[:200])

Mounted at /content/drive
Select all PDF files to upload...


Saving 2026 ICAP report.pdf to 2026 ICAP report.pdf
Saving 2025 carbon market report.pdf to 2025 carbon market report.pdf
Saving 2025 ICAP report.pdf to 2025 ICAP report.pdf
✓ Saved: /content/ets_pdfs/2026_ICAP_report.pdf | source=ICAP, year=2026, region=EU
✓ Saved: /content/ets_pdfs/2025_carbon_market_report.pdf | source=CarbonMarket, year=2025, region=EU
✓ Saved: /content/ets_pdfs/2025_ICAP_report.pdf | source=ICAP, year=2025, region=EU
ICAP (2026): kept 231 pages, skipped 62
CarbonMarket (2025): kept 49 pages, skipped 0
ICAP (2025): kept 207 pages, skipped 65

Total EU-relevant pages: 487
Total chunks after chunking: 1773
Chunks after keyword filter: 1546
Chunks after BM25 filter:   400
Stored batch 1 / 8
Stored batch 2 / 8
Stored batch 3 / 8
Stored batch 4 / 8
Stored batch 5 / 8
Stored batch 6 / 8
Stored batch 7 / 8
Stored batch 8 / 8

Done. Total chunks in ChromaDB: 400

--- Result 1 ---
Source: CarbonMarket, Page: 19
improve the transparency of the EU carbon market. ESMA was task

####Reload Embedded Chunks in ChromaDB from Drive

In [ ]:
# ── Reload ChromaDB from Drive (use this in new sessions) ──
from google.colab import drive
drive.mount('/content/drive')

from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import chromadb

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/ets_rag/chromadb"
)
collection = client.get_collection(
    name="ets_knowledge_base",
    embedding_function=embedding_fn
)

print(f"✓ Loaded {collection.count()} chunks from Drive")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Loaded 400 chunks from Drive


###Step 3: LLM Query Rewriter
**Objective:**
Translate the slider state (structured numbers) into a semantically rich retrieval query that ChromaDB can use to find relevant chunks.  

In [ ]:
def rewrite_query(slider_state: dict) -> str:
    prompt = f"""You are a financial analyst assistant specialising in EU carbon markets.

A user has adjusted the following EU ETS market parameters:
{slider_state}

Your task: write ONE concise retrieval query (max 40 words) that captures
the key policy and market dynamics implied by these parameter changes.
The query will be used to search a knowledge base of EU ETS policy documents.

Rules:
- Use domain vocabulary: EUA, MSR, auction, allowance, cap, compliance cost, etc.
- Do NOT mention numbers or percentages from the input directly
- Focus on the policy mechanisms and market effects implied by the changes
- Output the query only, no explanation

Query:"""

    # Retry up to 3 times
    for attempt in range(3):
        try:
            text = llm_call(prompt)

            # Validate: must be at least 10 words and contain ETS vocabulary
            ETS_VOCAB = ["carbon", "ets", "eua", "allowance", "auction",
                         "msr", "emission", "compliance", "price", "cap"]
            words = text.lower().split()

            if len(words) >= 10 and any(v in text.lower() for v in ETS_VOCAB):
                return text
            else:
                print(f"Attempt {attempt+1}: query failed validation, retrying...")

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")

    # Fallback: construct query from slider keys directly
    fallback = ("EU ETS carbon price allowance auction supply demand "
                "MSR compliance cost policy impact")
    print(f"Using fallback query: {fallback}")
    return fallback


# ── Test with multiple scenarios ─────────────────────────
test_cases = [
    {"price_change": 20,  "auction_volume": -10, "free_allocation": 0},
    {"price_change": -30, "auction_volume": 0,   "free_allocation": 10},
    {"price_change": 0,   "auction_volume": -25, "free_allocation": -5},
]

for state in test_cases:
    query = rewrite_query(state)
    print(f"\nSlider: {state}")
    print(f"Query:  {query}")


Slider: {'price_change': 20, 'auction_volume': -10, 'free_allocation': 0}
Query:  EUA price increase drivers when auction supply volumes decline, impacts on compliance costs, market liquidity effects, secondary market dynamics, MSR intake triggers, and allowance scarcity signals under reduced primary market availability.

Slider: {'price_change': -30, 'auction_volume': 0, 'free_allocation': 10}
Query:  EU ETS free allocation increase policy impacts on EUA price decline, industrial competitiveness, carbon leakage risk, auction revenue reduction, compliance costs for installations, and market supply-demand dynamics under MSR mechanism.

Slider: {'price_change': 0, 'auction_volume': -25, 'free_allocation': -5}
Query:  EU ETS auction volume reduction combined with decreased free allocation impact on allowance supply, MSR intake dynamics, compliance costs for installations, and EUA price formation under tightened cap conditions.


In [ ]:
# ── Chain test: slider → rewrite → retrieve ──────────────
test_slider_state = {
    "price_change": 20,
    "auction_volume": -10,
    "free_allocation": 0
}

# Step 1: rewrite slider state into natural language query
query = rewrite_query(test_slider_state)
print(f"Rewritten query:\n{query}\n")

# Step 2: feed rewritten query into ChromaDB
results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"region": "EU"}
)

# Step 3: print retrieved chunks
for i, doc in enumerate(results["documents"][0]):
    print(f"--- Result {i+1} ---")
    print(f"Source: {results['metadatas'][0][i]['source']}, "
          f"Page: {results['metadatas'][0][i]['page']}")
    print(doc[:300])
    print()

Rewritten query:
EUA price increase dynamics when auction volumes decline while free allocation remains unchanged, impacts on compliance costs, secondary market liquidity, MSR intake thresholds, and allowance scarcity effects on industrial sectors

--- Result 1 ---
Source: ICAP, Page: 60
1,000 tonne-kilometers (same benchmark as in the EU ETS).
Auctioning: Allowances that were not allocated for free were auctioned. Auctions took place two 
or three times a year, depending on available auction volumes. Since January 2020, auctions are 
open to entities covered by the Swiss ETS and th

--- Result 2 ---
Source: CarbonMarket, Page: 12
11 
 
advance the clean energy transformation and boost energy security. By 30 June 2025, EUR 11.5 billion 
was raised for the RRF - REPowerEU with 164 710 000 allowances auctioned. 
Figure 2 gives an overview of auction clearing prices in the EU carbon market in 2024 and the first half 
of 2025. Th

--- Result 3 ---
Source: ICAP, Page: 197
applied for those 

###Step 4: Retrieval & Reranking
**Objective:**
Retrieve top-k chunks from ChromaDB, then rerank with a cross-encoder to push the most relevant chunks to the top and filter noise.Using `cross-encoder/ms-marco-MiniLM-L-6-v2`from sentence-transformer to rerank chunks. The hyperparameters `initial_k=10`,`final_k=3`,`rerank_thershold=0.0`.

In [ ]:
from sentence_transformers import CrossEncoder

# ── Load reranker (local, no API cost) ───────────────────
print("Loading reranker model...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker ready.")

# ── Combined retrieve + rerank function ──────────────────
def retrieve_and_rerank(query: str,
                        collection,
                        initial_k: int = 10,
                        final_k: int = 3,
                        rerank_threshold: float = 0.0,
                        verbose: bool = False) -> list:
    """
    Parameters
    ----------
    query             : rewritten natural language query from Step 3
    collection        : ChromaDB collection from Step 2
    initial_k         : chunks retrieved by vector similarity before reranking
    final_k           : chunks returned after reranking
    rerank_threshold  : minimum cross-encoder score to keep a chunk (0.0 = keep all)
    verbose           : True = print before/after comparison, False = silent

    Returns
    -------
    list of dicts with keys: text, metadata, vector_distance, rerank_score
    """

    # ── Stage 1: vector retrieval ────────────────────────
    raw_results = collection.query(
        query_texts=[query],
        n_results=initial_k,
        where={"region": "EU"}
    )

    docs      = raw_results["documents"][0]
    metadatas = raw_results["metadatas"][0]
    distances = raw_results["distances"][0]

    if verbose:
        print("=" * 60)
        print("BEFORE RERANKING (vector similarity order)")
        print("=" * 60)
        for i in range(len(docs)):
            print(f"Rank {i+1} | Distance: {distances[i]:.4f} | "
                  f"Source: {metadatas[i]['source']}, "
                  f"Page: {metadatas[i]['page']}")
            print(docs[i][:150])
            print()

    # ── Stage 2: cross-encoder reranking ─────────────────
    pairs  = [(query, doc) for doc in docs]
    scores = reranker.predict(pairs)

    scored_chunks = [
        {
            "text":            docs[i],
            "metadata":        metadatas[i],
            "vector_distance": float(distances[i]),
            "rerank_score":    float(scores[i])
        }
        for i in range(len(docs))
    ]

    # Sort by rerank score descending
    scored_chunks.sort(key=lambda x: x["rerank_score"], reverse=True)

    # Apply minimum quality threshold
    scored_chunks = [c for c in scored_chunks
                     if c["rerank_score"] >= rerank_threshold]

    if verbose:
        print("=" * 60)
        print("AFTER RERANKING (cross-encoder score order)")
        print("=" * 60)
        for i, chunk in enumerate(scored_chunks[:final_k]):
            print(f"Rank {i+1} | "
                  f"Rerank score: {chunk['rerank_score']:.4f} | "
                  f"Vector distance: {chunk['vector_distance']:.4f} | "
                  f"Source: {chunk['metadata']['source']}, "
                  f"Page: {chunk['metadata']['page']}")
            print(chunk["text"][:150])
            print()

    return scored_chunks[:final_k]


# ── Test: verbose=True to see before/after ───────────────
test_query = rewrite_query({
    "price_change": 20,
    "auction_volume": -10,
    "free_allocation": 0
})

print(f"Query: {test_query}\n")

reranked = retrieve_and_rerank(
    query=test_query,
    collection=collection,
    initial_k=10,
    final_k=3,
    rerank_threshold=0.0,
    verbose=True          # ← shows before/after, use for report screenshots
)

# ── Summary: final chunks going into prompt ───────────────
print("=" * 60)
print("FINAL CHUNKS FOR PROMPT")
print("=" * 60)
for i, chunk in enumerate(reranked):
    print(f"Chunk {i+1} | Score: {chunk['rerank_score']:.4f} | "
          f"Source: {chunk['metadata']['source']}, "
          f"Page: {chunk['metadata']['page']}")
    print(chunk["text"][:200])
    print()

Loading reranker model...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker ready.
Query: EUA price increase drivers when auction volumes decline, supply-demand imbalance effects on compliance costs, MSR intake triggers, secondary market liquidity impact, and allowance scarcity under reduced primary market supply.

BEFORE RERANKING (vector similarity order)
Rank 1 | Distance: 0.4368 | Source: ICAP, Page: 50
MARKET REGULATION
MARKET STABILITY PROVISIONS
EU ETS 2 MARKET STABILITY RESERVE (MSR)
Instrument type: Quantity-based instrument
Over the course of th

Rank 2 | Distance: 0.4399 | Source: ICAP, Page: 60
1,000 tonne-kilometers (same benchmark as in the EU ETS).
Auctioning: Allowances that were not allocated for free were auctioned. Auctions took place 

Rank 3 | Distance: 0.4432 | Source: ICAP, Page: 10
uncertainty, regulatory adjustments, and shifting market sentiment, 
while jurisdictions with fixed price trajectories, like Germany and Canada, 
expe

Rank 4 | Distance: 0.4463 | Source: CarbonMarket, Page: 12
11 
 
advance the clean energy transfor

###Step 5: Simulation Engine & ROE Calculator
**Objective:**
Build the pure math engine that translates slider inputs into a ROE delta. No LLM, no RAG — clean deterministic calculation. The engine is deliberately isolated from the RAG pipeline for one reason: **LLM hallucination must never contaminate financial calculations.**
If you let the LLM compute ROE, it might produce plausible-sounding but mathematically wrong numbers. By separating the engine, you guarantee:
*   *Numbers* are always *deterministic* and *reproducible*
*   *RAG* context influences the *interpretation* of numbers, never the numbers themselves


In [ ]:
#ROE = Net Income / Equity

#Net Income = Revenue - Carbon Cost - Operating Cost

#Carbon Cost = Baseline Emissions × Carbon Price
#Revenue = Baseline Revenue × (1 + price_change/100)

In [ ]:
#First, create simulation_config.json.
import json
from pathlib import Path

config = {
    "baseline": {
        "revenue": 10000000,        # €10M revenue
        "operating_cost": 6000000,  # €6M operating cost (60% margin)
        "baseline_emissions": 50000,# 50,000 tCO₂e
        "baseline_carbon_price": 65,# €65/tonne EUA
        "equity": 3000000           # €3M equity
    },
    "coefficients": {
        "price_change": {
            "revenue_sensitivity": 1.0,
            "comment": "1% price change → 1% revenue change"
        },
        "auction_volume": {
            "carbon_price_sensitivity": -0.3,
            "comment": "1% auction volume reduction → 0.3% carbon price increase"
        },
        "free_allocation": {
            "carbon_cost_sensitivity": -0.5,
            "comment": "1% more free allocation → 0.5% carbon cost reduction"
        }
    }
}

# Write to file
config_path = Path("/content/simulation_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✓ simulation_config.json created at {config_path}")

# Verify by reading it back
with open(config_path, "r") as f:
    loaded = json.load(f)
print(f"✓ Verified: {list(loaded.keys())}")

✓ simulation_config.json created at /content/simulation_config.json
✓ Verified: ['baseline', 'coefficients']


In [ ]:
# ── Load config ──────────────────────────────────────────
with open("simulation_config.json", "r") as f:
    config = json.load(f)

def calculate_roe(slider_state: dict, config: dict) -> dict:
    """
    Pure math ROE calculator. No LLM, no RAG.

    Parameters
    ----------
    slider_state : dict with keys price_change, auction_volume, free_allocation
    config       : loaded simulation_config.json

    Returns
    -------
    dict with baseline_roe, new_roe, roe_delta, and breakdown
    """
    b    = config["baseline"]
    coef = config["coefficients"]

    # ── Baseline ROE ─────────────────────────────────────
    baseline_carbon_cost = (b["baseline_emissions"] *
                            b["baseline_carbon_price"])
    baseline_net_income  = (b["revenue"] -
                            baseline_carbon_cost -
                            b["operating_cost"])
    baseline_roe = baseline_net_income / b["equity"]

    # ── Adjusted values from sliders ─────────────────────
    # Revenue effect from price change
    new_revenue = b["revenue"] * (
        1 + slider_state["price_change"] / 100
        * coef["price_change"]["revenue_sensitivity"]
    )

    # Carbon price effect from auction volume change
    carbon_price_multiplier = (
        1 - slider_state["auction_volume"] / 100
        * coef["auction_volume"]["carbon_price_sensitivity"]
    )
    new_carbon_price = b["baseline_carbon_price"] * carbon_price_multiplier

    # Carbon cost effect from free allocation change
    carbon_cost_multiplier = (
        1 + slider_state["free_allocation"] / 100
        * coef["free_allocation"]["carbon_cost_sensitivity"]
    )
    new_carbon_cost = (b["baseline_emissions"] *
                       new_carbon_price *
                       carbon_cost_multiplier)

    # ── New ROE ──────────────────────────────────────────
    new_net_income = (new_revenue -
                      new_carbon_cost -
                      b["operating_cost"])
    new_roe = new_net_income / b["equity"]

    roe_delta = new_roe - baseline_roe

    return {
        "baseline_roe":       round(baseline_roe, 4),
        "new_roe":            round(new_roe, 4),
        "roe_delta":          round(roe_delta, 4),
        "breakdown": {
            "baseline_revenue":     b["revenue"],
            "new_revenue":          round(new_revenue, 2),
            "baseline_carbon_cost": round(baseline_carbon_cost, 2),
            "new_carbon_cost":      round(new_carbon_cost, 2),
            "new_carbon_price":     round(new_carbon_price, 2),
            "new_net_income":       round(new_net_income, 2),
        }
    }

def sensitivity_analysis(config: dict) -> None:
    """
    Tests each slider independently at +10% and -10%
    to show which lever has the most ROE impact.
    Useful for the report's analysis section.
    """
    sliders = ["price_change", "auction_volume", "free_allocation"]
    base    = {"price_change": 0, "auction_volume": 0, "free_allocation": 0}

    print("=" * 55)
    print("SENSITIVITY ANALYSIS — ROE delta per slider")
    print("=" * 55)

    for slider in sliders:
        for delta in [+10, -10]:
            state = base.copy()
            state[slider] = delta
            result = calculate_roe(state, config)
            print(f"{slider:20s} {delta:+3d}% → "
                  f"ROE delta: {result['roe_delta']:+.2%}")
        print()

# ── Test ─────────────────────────────────────────────────
test_slider = {
    "price_change":   20,
    "auction_volume": -10,
    "free_allocation": 0
}

result = calculate_roe(test_slider, config)

print(f"Baseline ROE : {result['baseline_roe']:.2%}")
print(f"New ROE      : {result['new_roe']:.2%}")
print(f"ROE Delta    : {result['roe_delta']:.2%}")
print(f"\nBreakdown:")
for k, v in result["breakdown"].items():
    print(f"  {k}: {v}")

print()
sensitivity_analysis(config)

Baseline ROE : 25.00%
New ROE      : 94.92%
ROE Delta    : 69.92%

Breakdown:
  baseline_revenue: 10000000
  new_revenue: 12000000.0
  baseline_carbon_cost: 3250000
  new_carbon_cost: 3152500.0
  new_carbon_price: 63.05
  new_net_income: 2847500.0

SENSITIVITY ANALYSIS — ROE delta per slider
price_change         +10% → ROE delta: +33.33%
price_change         -10% → ROE delta: -33.33%

auction_volume       +10% → ROE delta: -3.25%
auction_volume       -10% → ROE delta: +3.25%

free_allocation      +10% → ROE delta: +5.42%
free_allocation      -10% → ROE delta: -5.42%



###Step 6: Prompt Assembly & Report Generation
**Objective:**
Assemble ROE results + retrieved chunks into a prompt, call Gemini to generate narrative sections, then export a fully formatted .docx report`'train_ROE_report.docx'` and a short textual summary. The report would consist of 4 sections within 3 pages:  
`action review`: review actions that have been done to sliders. It includes subjective, objective, direction and the number. The action should be displayed in bullet point one by one.  
`brief overview`: a brief summary for the whole report within 2 paragraphs and 150 words per paragraph.  
`analysis`: include two parts, one is to analyse each action, and the other one is a comprehensive analysis with a comparison table about numerical changes and their impacts on ROE, and a ROE formula to show the calculation.  
`summary`: one paragraph within 400 words.  

This step has two parts:

*   Part A: Call Gemini once to generate the three narrative sections (Action Review bullets, Brief Overview, Analysis text, Summary) as structured JSON
*   Part B: Use python-docx to assemble the final formatted document with exact font/layout specifications  

I use JSON output from Gemini so the narrative slots cleanly into the Word template without any string parsing mess.


In [ ]:
import json,re
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# ── Part A: Generate narrative via Gemini ────────────────
def generate_narrative(slider_state: dict,
                       roe_result: dict,
                       reranked_chunks: list) -> dict:

    context = "\n\n".join([
        f"[Source: {c['metadata']['source']}, "
        f"Page {c['metadata']['page']}]\n{c['text']}"
        for c in reranked_chunks
    ])

    b = roe_result["breakdown"]

    prompt = f"""You are a senior financial analyst specialising in EU carbon markets.

You have been given:
1. SLIDER ACTIONS:
{json.dumps(slider_state, indent=2)}

2. ROE CALCULATION RESULTS:
- Baseline ROE: {roe_result['baseline_roe']:.2%}
- New ROE: {roe_result['new_roe']:.2%}
- ROE Delta: {roe_result['roe_delta']:.2%}
- Baseline Revenue: €{b['baseline_revenue']:,}
- New Revenue: €{b['new_revenue']:,}
- Baseline Carbon Cost: €{b['baseline_carbon_cost']:,}
- New Carbon Cost: €{b['new_carbon_cost']:,}
- New Carbon Price: €{b['new_carbon_price']}/tonne
- New Net Income: €{b['new_net_income']:,}

3. RETRIEVED EU ETS POLICY CONTEXT:
{context}

Generate a structured report in JSON format with EXACTLY these keys:

{{
  "action_review": [
    {{
      "action": "slider name",
      "direction": "increased/decreased",
      "magnitude": "X%",
      "objective": "what this represents in EU ETS context",
      "effect": "immediate financial effect"
    }}
  ],
  "brief_overview": {{
    "paragraph_1": "first paragraph, max 150 words, high-level summary",
    "paragraph_2": "second paragraph, max 150 words, policy context from retrieved chunks"
  }},
  "analysis": {{
    "per_action": [
      {{
        "action": "slider name",
        "analysis": "2-3 sentences analysing this specific action impact"
      }}
    ],
    "comprehensive": "3-4 sentences comprehensive analysis grounded in retrieved policy context"
  }},
  "summary": "one paragraph, max 400 words, strategic implications and recommendations",
  "short_summary": "2-3 sentences maximum. State what actions were taken and what specific EU ETS policy mechanisms make this significant. Ground this in the retrieved context. This is used for baseline comparison."
}}

Rules:
- Use professional financial language
- Ground policy observations in the retrieved context
- Do NOT invent numbers — use only the figures provided above
- Output valid JSON only, no markdown fences, no preamble
"""

    text = llm_call(prompt)

    if text.startswith("```"):
        text = re.sub(r"^```[a-z]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text)

    return json.loads(text)


# ── Part B: Build .docx report ───────────────────────────
def add_page_break(doc):
    paragraph = doc.add_paragraph()
    run = paragraph.runs
    if run:
        run[0].add_break()
    else:
        run = paragraph.add_run()
    run.add_break(WD_ALIGN_PARAGRAPH.LEFT)
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
    br = OxmlElement("w:br")
    br.set(qn("w:type"), "page")
    paragraph.runs[-1]._r.append(br)

def set_font(run, size: int, bold: bool = False):
    run.font.name   = "Arial"
    run.font.size   = Pt(size)
    run.font.bold   = bold

def add_heading(doc, text: str, size: int):
    para = doc.add_paragraph()
    para.alignment = WD_ALIGN_PARAGRAPH.LEFT
    run = para.add_run(text)
    set_font(run, size, bold=True)
    return para

def add_body(doc, text: str):
    para = doc.add_paragraph()
    run  = para.add_run(text)
    set_font(run, 11, bold=False)
    return para

def prevent_table_split(table):
    """Forces the entire table to stay on one page."""
    for row in table.rows:
        tr = row._tr
        trPr = tr.get_or_add_trPr()
        cantSplit = OxmlElement("w:cantSplit")
        cantSplit.set(qn("w:val"), "1")
        trPr.append(cantSplit)

def build_report(slider_state: dict,
                 roe_result: dict,
                 reranked_chunks: list,
                 output_path: str = "/content/ROE_Report.docx"):

    # Generate narrative
    print("Generating narrative via Gemini...")
    narrative = generate_narrative(slider_state, roe_result, reranked_chunks)
    print("Narrative generated. Building document...")

    doc = Document()

    # ── Set narrow margins ───────────────────────────────
    for section in doc.sections:
        section.top_margin    = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin   = Inches(1.2)
        section.right_margin  = Inches(1.2)

    # ════════════════════════════════════════════════════
    # PAGE 1: Cover
    # ════════════════════════════════════════════════════
    cover = doc.add_paragraph()
    cover.alignment = WD_ALIGN_PARAGRAPH.CENTER
    # Push title to vertical centre with spacing
    cover.paragraph_format.space_before = Pt(200)
    run = cover.add_run("ROE Report")
    set_font(run, 26, bold=True)

    # Page break → content starts page 2
    page_break_para = doc.add_paragraph()
    run_br = page_break_para.add_run()
    br = OxmlElement("w:br")
    br.set(qn("w:type"), "page")
    run_br._r.append(br)

    # ════════════════════════════════════════════════════
    # PAGE 2+: Content
    # ════════════════════════════════════════════════════

    # ── 1. Action Review ─────────────────────────────────
    add_heading(doc, "1. Action Review", 13)
    for item in narrative["action_review"]:
        bullet = doc.add_paragraph(style="List Bullet")
        run    = bullet.add_run(
            f"{item['action']}: {item['direction']} by {item['magnitude']} — "
            f"{item['objective']} | Effect: {item['effect']}"
        )
        set_font(run, 11)

    doc.add_paragraph()  # spacer

    # ── 2. Brief Overview ────────────────────────────────
    add_heading(doc, "2. Brief Overview", 13)
    add_body(doc, narrative["brief_overview"]["paragraph_1"])
    doc.add_paragraph()
    add_body(doc, narrative["brief_overview"]["paragraph_2"])

    doc.add_paragraph()  # spacer

    # ── 3. Analysis ──────────────────────────────────────
    add_heading(doc, "3. Analysis", 13)

    # 3.1 Per-action analysis
    add_heading(doc, "3.1 Per-Action Analysis", 12)
    for item in narrative["analysis"]["per_action"]:
        add_heading(doc, f"• {item['action']}", 11)
        add_body(doc, item["analysis"])

    doc.add_paragraph()

    # 3.2 Comprehensive analysis
    add_heading(doc, "3.2 Comprehensive Analysis", 12)
    add_body(doc, narrative["analysis"]["comprehensive"])

    doc.add_paragraph()

    # 3.2 Table: before/after
    add_heading(doc, "Financial Impact Table", 12)
    b = roe_result["breakdown"]

    table = doc.add_table(rows=6, cols=4)
    table.style = "Table Grid"
    prevent_table_split(table)

    headers = ["Metric", "Before", "After", "Impact on ROE"]
    for i, h in enumerate(headers):
        cell = table.rows[0].cells[i]
        run  = cell.paragraphs[0].add_run(h)
        set_font(run, 11, bold=True)

    rows_data = [
        ["Revenue",
         f"€{b['baseline_revenue']:,}",
         f"€{b['new_revenue']:,}",
         f"+\u20ac{b['new_revenue'] - b['baseline_revenue']:,.0f}"],
        ["Carbon Cost",
         f"\u20ac{b['baseline_carbon_cost']:,}",
         f"\u20ac{b['new_carbon_cost']:,}",
         f"\u20ac{b['new_carbon_cost'] - b['baseline_carbon_cost']:,.0f} saved"
         if b['new_carbon_cost'] < b['baseline_carbon_cost']
         else f"+\u20ac{b['new_carbon_cost'] - b['baseline_carbon_cost']:,.0f}"],
        ["Net Income",
         f"\u20ac{b['baseline_revenue'] - b['baseline_carbon_cost'] - 6000000:,}",
         f"\u20ac{b['new_net_income']:,}",
         f"+\u20ac{b['new_net_income'] - (b['baseline_revenue'] - b['baseline_carbon_cost'] - 6000000):,.0f}"],
        ["ROE",
         f"{roe_result['baseline_roe']:.2%}",
         f"{roe_result['new_roe']:.2%}",
         f"+{roe_result['roe_delta']:.2%}"],
        ["Carbon Price",
         f"\u20ac{b['baseline_carbon_cost'] / 50000:.2f}/t",
         f"\u20ac{b['new_carbon_price']:.2f}/t",
         "—"],
    ]

    for row_idx, row_data in enumerate(rows_data):
        for col_idx, value in enumerate(row_data):
            cell = table.rows[row_idx + 1].cells[col_idx]
            run  = cell.paragraphs[0].add_run(value)
            set_font(run, 11)

    doc.add_paragraph()

    # ROE formula
    add_heading(doc, "ROE Calculation", 12)
    formula_text = (
        f"ROE  =  Net Income / Equity\n"
        f"     =  \u20ac{b['new_net_income']:,} / \u20ac3,000,000\n"
        f"     =  {roe_result['new_roe']:.2%}"
    )
    para = doc.add_paragraph()
    run  = para.add_run(formula_text)
    set_font(run, 11)
    run.font.name = "Courier New"  # monospace for formula alignment

    doc.add_paragraph()

    # ── 4. Summary ───────────────────────────────────────
    add_heading(doc, "4. Summary", 13)
    add_body(doc, narrative["summary"])

    # ── Save ─────────────────────────────────────────────
    doc.save(output_path)
    print(f"\n✓ Report saved to {output_path}")
    return output_path


# ── Run it ───────────────────────────────────────────────
output_path = build_report(
    slider_state    = test_slider,
    roe_result      = result,
    reranked_chunks = reranked,
    output_path     = "/content/train_ROE_Report.docx"
)

# Download in Colab
from google.colab import files
files.download(output_path)

Generating narrative via Gemini...
Narrative generated. Building document...

✓ Report saved to /content/train_ROE_Report.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Call generate_narrative and store the result
enhanced_narrative = generate_narrative(
    slider_state    = test_slider,
    roe_result      = result,
    reranked_chunks = reranked
)

print(f"Short summary:\n{enhanced_narrative['short_summary']}")

Short summary:
Carbon price increased 20% to €63.05/tonne while auction volume decreased 10%, generating exceptional ROE expansion to 94.92%. This price level approaches the EUR 45 MSR intervention threshold where the Commission's November 2025 proposal would double allowance releases to moderate excessive price increases, creating significant policy risk to sustaining these returns.


###Step 7: Baseline Pipeline
**Objective:**
Build the vanilla LLM pipeline and compare it with RAG-augmented pipeline on the short summary they generated due to the limited quota for free LLM and API key. Vanilla LLM can only repeat actions, while RAG-augmented pipleline can do contextual analysis.

In [ ]:
def run_baseline_pipeline(slider_state: dict,
                          roe_result: dict) -> tuple:

    # Step 1: no query rewriting — just stringify the slider
    raw_query = (f"carbon price change {slider_state['price_change']}% "
                 f"auction volume {slider_state['auction_volume']}% "
                 f"free allocation {slider_state['free_allocation']}%")

    # Step 2: simple vector retrieval, no metadata filter, no reranking
    raw_results = collection.query(
        query_texts=[raw_query],
        n_results=3
    )
    raw_chunks = raw_results["documents"][0]

    # Step 3: vanilla prompt
    context = "\n\n".join(raw_chunks)

    prompt = f"""You are a financial analyst.

Slider actions: {slider_state}
ROE changed from {roe_result['baseline_roe']:.2%} to {roe_result['new_roe']:.2%}.

Context:
{context}

Write a 2-3 sentence summary of what happened and why it matters.
"""
    summary = llm_call(prompt)
    return summary, raw_chunks  # ← this line was missing


# ── Run both pipelines ────────────────────────────────────
print("=" * 55)
print("BASELINE: simple retrieval + vanilla prompt")
print("=" * 55)
baseline_summary, baseline_chunks = run_baseline_pipeline(
    slider_state = test_slider,
    roe_result   = result
)
print(f"\nBaseline summary:\n{baseline_summary}")

print("\n" + "=" * 55)
print("ENHANCED: query rewriting + reranking + grounded prompt")
print("=" * 55)
enhanced_short_summary = enhanced_narrative["short_summary"]
print(f"\nEnhanced summary:\n{enhanced_short_summary}")

print("\n" + "=" * 55)
print("COMPARISON")
print("=" * 55)
print(f"\nBaseline chunks retrieved:")
for c in baseline_chunks:
    print(f"  {c[:80]}...")
print(f"\nEnhanced chunks retrieved:")
for c in reranked:
    print(f"  {c['text'][:80]}...")

BASELINE: simple retrieval + vanilla prompt

Baseline summary:
# Summary

With a 20% price increase and 10% decrease in auction volume while free allocation remained unchanged, return on equity (ROE) surged from 25% to 94.92%, indicating dramatically improved profitability for market participants. This matters because the combination of higher allowance prices and reduced auction supply creates scarcity in the market, significantly increasing the value of existing allowance holdings—particularly benefiting entities with substantial free allocations who can now sell surplus allowances at elevated prices or reduce compliance costs.

ENHANCED: query rewriting + reranking + grounded prompt

Enhanced summary:
Carbon price increased 20% to €63.05/tonne while auction volume decreased 10%, generating exceptional ROE expansion to 94.92%. This price level approaches the EUR 45 MSR intervention threshold where the Commission's November 2025 proposal would double allowance releases to moderate exc

###Step 8: ipywidgets Dashboard
**Objective:**
Wire all pipeline components into an interactive dashboard with sliders, a generate button, and an output area that displays the report and downloads the .docx. When user move sliders and click `generate report` button, a textual short summary will show up, and a report `ROE_Report.docx`would be generated and downloaded automatically.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Sliders ───────────────────────────────────────────────
slider_price = widgets.IntSlider(
    value=0, min=-50, max=50, step=5,
    description="Price Change %:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

slider_auction = widgets.IntSlider(
    value=0, min=-50, max=50, step=5,
    description="Auction Volume %:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

slider_allocation = widgets.IntSlider(
    value=0, min=-50, max=50, step=5,
    description="Free Allocation %:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

# ── Buttons ───────────────────────────────────────────────
btn_generate = widgets.Button(
    description="Generate Report",
    button_style="primary",
    layout=widgets.Layout(width="200px", height="40px")
)

btn_reset = widgets.Button(
    description="Reset Sliders",
    button_style="warning",
    layout=widgets.Layout(width="200px", height="40px")
)

# ── Output area ───────────────────────────────────────────
out = widgets.Output()

# ── ROE display ───────────────────────────────────────────
roe_display = widgets.HTML(
    value="<b>Baseline ROE:</b> — &nbsp;&nbsp; <b>New ROE:</b> — &nbsp;&nbsp; <b>Delta:</b> —"
)

# ── Live ROE update on slider move ────────────────────────
def update_roe_display(change):
    slider_state = {
        "price_change":   slider_price.value,
        "auction_volume": slider_auction.value,
        "free_allocation": slider_allocation.value
    }
    r = calculate_roe(slider_state, config)
    roe_display.value = (
        f"<b>Baseline ROE:</b> {r['baseline_roe']:.2%} &nbsp;&nbsp; "
        f"<b>New ROE:</b> {r['new_roe']:.2%} &nbsp;&nbsp; "
        f"<b>Delta:</b> "
        f"<span style='color:{'green' if r['roe_delta'] >= 0 else 'red'}'>"
        f"{r['roe_delta']:+.2%}</span>"
    )

slider_price.observe(update_roe_display, names="value")
slider_auction.observe(update_roe_display, names="value")
slider_allocation.observe(update_roe_display, names="value")

# ── Generate button handler ───────────────────────────────
def on_generate(b):
    with out:
        clear_output(wait=True)

        slider_state = {
            "price_change":    slider_price.value,
            "auction_volume":  slider_auction.value,
            "free_allocation": slider_allocation.value
        }

        print("Step 1/4: Calculating ROE...")
        roe_result = calculate_roe(slider_state, config)
        print(f"  Baseline ROE: {roe_result['baseline_roe']:.2%} → "
              f"New ROE: {roe_result['new_roe']:.2%} "
              f"(Δ {roe_result['roe_delta']:+.2%})")

        print("Step 2/4: Rewriting query...")
        query = rewrite_query(slider_state)
        print(f"  Query: {query}")

        print("Step 3/4: Retrieving and reranking...")
        reranked_chunks = retrieve_and_rerank(
            query      = query,
            collection = collection,
            initial_k  = 10,
            final_k    = 3,
            verbose    = False
        )
        print(f"  Retrieved {len(reranked_chunks)} chunks")

        print("Step 4/4: Generating report...")
        narrative = generate_narrative(
            slider_state    = slider_state,
            roe_result      = roe_result,
            reranked_chunks = reranked_chunks
        )

        # Build and download .docx
        output_path = build_report(
            slider_state    = slider_state,
            roe_result      = roe_result,
            reranked_chunks = reranked_chunks,
            output_path     = "/content/ROE_Report.docx"
        )

        print("\n" + "=" * 50)
        print("REPORT SUMMARY")
        print("=" * 50)
        print(f"\n{narrative['short_summary']}")
        print("\n" + "=" * 50)
        print("Brief Overview")
        print("=" * 50)
        print(f"\n{narrative['brief_overview']['paragraph_1']}")
        print(f"\n{narrative['brief_overview']['paragraph_2']}")

        # Auto-download
        from google.colab import files
        files.download(output_path)
        print("\n✓ Report downloaded.")

# ── Reset button handler ──────────────────────────────────
def on_reset(b):
    slider_price.value     = 0
    slider_auction.value   = 0
    slider_allocation.value = 0
    with out:
        clear_output()
    roe_display.value = (
        "<b>Baseline ROE:</b> — &nbsp;&nbsp; "
        "<b>New ROE:</b> — &nbsp;&nbsp; "
        "<b>Delta:</b> —"
    )

btn_generate.on_click(on_generate)
btn_reset.on_click(on_reset)

# ── Layout and display ────────────────────────────────────
title = widgets.HTML(
    value="""
    <p style='color:#7f8c8d'>
    <b>How to use:</b><br>
    1. Adjust the sliders below to set market parameters<br>
    2. Click <b>Generate Report</b> to produce a policy-grounded ROE analysis<br>
    3. A short summary will appear below and a full .docx report will download automatically
    </p>
    """
)

subtitle = widgets.HTML(
    value="<p style='color:#7f8c8d'>Adjust market parameters and generate a policy-grounded ROE analysis report.</p>"
)

button_row = widgets.HBox([btn_generate, btn_reset])

dashboard = widgets.VBox([
    title,
    subtitle,
    widgets.HTML("<hr>"),
    slider_price,
    slider_auction,
    slider_allocation,
    widgets.HTML("<hr>"),
    roe_display,
    widgets.HTML("<br>"),
    button_row,
    widgets.HTML("<br>"),
    out
])

display(dashboard)

###Step 9: Evaluation
**Objective:**
Evaluate retrieval quality of baseline vs enhanced pipeline using embedding-based metrics only. No API calls needed.

I use two metrics that run purely on embeddings:


*   Context Precision:Measures retrieval noise.
*   Context Recall: Measures retrieval completeness.

Using Claude to generate a test set of 15 queries across 4 types. Run both pipelines, and compare scores.

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

# ── Load local embedding model ────────────────────────────
eval_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✓ Evaluation model loaded.")

# ── Metric functions with NaN guards ─────────────────────
def cosine_sim(a, b):
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(np.dot(a, b) / norm)

def context_precision_manual(contexts: list, ground_truth: str) -> float:
    if not contexts or not any(c.strip() for c in contexts):
        return 0.0
    gt_emb = eval_model.encode(ground_truth)
    scores = [cosine_sim(eval_model.encode(c), gt_emb) for c in contexts]
    return float(np.mean([s > 0.5 for s in scores]))

def context_recall_manual(contexts: list, ground_truth: str) -> float:
    if not contexts or not any(c.strip() for c in contexts):
        return 0.0
    gt_emb   = eval_model.encode(ground_truth)
    combined = " ".join(contexts).strip()
    if not combined:
        return 0.0
    comb_emb = eval_model.encode(combined)
    return cosine_sim(gt_emb, comb_emb)

# ── Test set ──────────────────────────────────────────────
TEST_QUERIES = [
    # Simple facts
    {"question": "What is the MSR activation threshold in the EU ETS?",
     "ground_truth": "The MSR is activated when the total number of allowances in circulation exceeds 833 million."},
    {"question": "What was the total EU ETS auction revenue in 2024?",
     "ground_truth": "The total revenue generated by the EU ETS in 2024 amounted to EUR 38.8 billion."},
    {"question": "What penalty do regulated entities pay for excess emissions?",
     "ground_truth": "Regulated entities pay EUR 100 per tCO2e, adjusted for inflation."},
    {"question": "What percentage of global emissions does the EU ETS cover?",
     "ground_truth": "The EU ETS covers approximately 40% of EU total emissions."},

    # Deep context
    {"question": "How does auction volume reduction affect carbon price through the MSR mechanism?",
     "ground_truth": "Reduced auction volumes tighten allowance supply, triggering MSR intervention which further reduces the number of allowances in circulation, putting upward pressure on EUA prices."},
    {"question": "What is the relationship between free allocation and carbon leakage risk?",
     "ground_truth": "Free allocation is used to protect industries at risk of carbon leakage, where production moves to regions with less stringent climate policies, undermining EU ETS effectiveness."},
    {"question": "How does CBAM interact with free allocation policy in the EU ETS?",
     "ground_truth": "CBAM is being phased in as free allocation is phased out for covered sectors, ensuring that imports face equivalent carbon costs to EU producers."},
    {"question": "What drove the decline in EU ETS emissions from power and industry in 2024?",
     "ground_truth": "EU ETS emissions from power and industry fell due to renewable energy expansion and energy efficiency improvements."},

    # Ambiguous
    {"question": "Is the carbon price high or low currently?",
     "ground_truth": "Carbon price levels depend on market conditions including supply from auctions, MSR interventions, and demand from compliance entities."},
    {"question": "Should companies buy more allowances now?",
     "ground_truth": "The decision depends on individual company emissions profiles, hedging strategies, and expectations about future carbon price trajectories under EU ETS policy."},
    {"question": "What happens to the ETS if emissions drop significantly?",
     "ground_truth": "If emissions drop significantly, allowance surplus increases, potentially triggering MSR absorption to stabilise prices and maintain the carbon price signal."},
    {"question": "How does the ETS affect industrial competitiveness?",
     "ground_truth": "The EU ETS affects competitiveness through compliance costs, but free allocation and CBAM are designed to mitigate carbon leakage and protect EU industry."},

    # Edge cases
    {"question": "What is the carbon price on Mars?",
     "ground_truth": "There is no carbon price on Mars. This question is outside the scope of the EU ETS knowledge base."},
    {"question": "How does the EU ETS affect cryptocurrency mining?",
     "ground_truth": "Cryptocurrency mining is not directly covered by the EU ETS, though energy-intensive operations may be indirectly affected through electricity prices."},
    {"question": "What was the EUA price in 1990?",
     "ground_truth": "The EU ETS did not exist in 1990. It was launched in 2005."},
]

# ── Pipeline functions ────────────────────────────────────
def run_enhanced_pipeline_eval(question: str) -> list:
    chunks = retrieve_and_rerank(
        query            = question,
        collection       = collection,
        initial_k        = 10,
        final_k          = 3,
        rerank_threshold = -10.0,  # accept all chunks
        verbose          = False
    )
    contexts = [c["text"] for c in chunks]
    if not contexts:
        print(f"  ⚠ Empty contexts for: {question[:55]}")
    return contexts

def run_baseline_pipeline_eval(question: str) -> list:
    raw_results = collection.query(
        query_texts=[question],
        n_results=3
    )
    contexts = raw_results["documents"][0]
    return contexts

# ── Run evaluation ────────────────────────────────────────
print("Evaluating enhanced pipeline...")
enhanced_precision, enhanced_recall = [], []

for item in TEST_QUERIES:
    contexts = run_enhanced_pipeline_eval(item["question"])
    enhanced_precision.append(
        context_precision_manual(contexts, item["ground_truth"])
    )
    enhanced_recall.append(
        context_recall_manual(contexts, item["ground_truth"])
    )
    print(f"  ✓ {item['question'][:55]}...")

print("\nEvaluating baseline pipeline...")
baseline_precision, baseline_recall = [], []

for item in TEST_QUERIES:
    contexts = run_baseline_pipeline_eval(item["question"])
    baseline_precision.append(
        context_precision_manual(contexts, item["ground_truth"])
    )
    baseline_recall.append(
        context_recall_manual(contexts, item["ground_truth"])
    )
    print(f"  ✓ {item['question'][:55]}...")

# ── Per-query results table ───────────────────────────────
print("\n" + "=" * 80)
print("  PER-QUERY RESULTS")
print("=" * 80)
print(f"{'#':<3} {'Type':<12} {'B.Prec':>8} {'E.Prec':>8} "
      f"{'B.Rec':>8} {'E.Rec':>8} {'Prec Δ':>8} {'Rec Δ':>8}")
print("-" * 80)

query_types = (["Simple fact"] * 4 + ["Deep context"] * 4 +
               ["Ambiguous"] * 4 + ["Edge case"] * 3)

for i in range(len(TEST_QUERIES)):
    bp = baseline_precision[i]
    ep = enhanced_precision[i]
    br = baseline_recall[i]
    er = enhanced_recall[i]
    print(f"{i+1:<3} {query_types[i]:<12} {bp:>8.3f} {ep:>8.3f} "
          f"{br:>8.3f} {er:>8.3f} "
          f"{ep-bp:>+8.3f} {er-br:>+8.3f}")

# ── Summary table ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY — BASELINE vs ENHANCED")
print("=" * 60)
print(f"{'Metric':<25} {'Baseline':>12} {'Enhanced':>12} {'Delta':>10}")
print("-" * 60)

summary = {
    "context_precision": (np.mean(baseline_precision),
                          np.mean(enhanced_precision)),
    "context_recall":    (np.mean(baseline_recall),
                          np.mean(enhanced_recall)),
}

for metric, (b, e) in summary.items():
    d = e - b
    print(f"{metric:<25} {b:>12.4f} {e:>12.4f} "
          f"{'+' + f'{d:.4f}' if d >= 0 else f'{d:.4f}':>10}")

print("=" * 60)

# ── Per query-type breakdown ──────────────────────────────
print("\n" + "=" * 60)
print("  BREAKDOWN BY QUERY TYPE")
print("=" * 60)
print(f"{'Type':<15} {'B.Prec':>10} {'E.Prec':>10} "
      f"{'B.Rec':>10} {'E.Rec':>10}")
print("-" * 60)

type_ranges = {
    "Simple fact":  range(0, 4),
    "Deep context": range(4, 8),
    "Ambiguous":    range(8, 12),
    "Edge case":    range(12, 15),
}

for qtype, rng in type_ranges.items():
    bp = np.mean([baseline_precision[i] for i in rng])
    ep = np.mean([enhanced_precision[i] for i in rng])
    br = np.mean([baseline_recall[i] for i in rng])
    er = np.mean([enhanced_recall[i] for i in rng])
    print(f"{qtype:<15} {bp:>10.3f} {ep:>10.3f} "
          f"{br:>10.3f} {er:>10.3f}")

print("=" * 60)

# ── Save to CSV ───────────────────────────────────────────
results_df = pd.DataFrame({
    "Question":           [q["question"] for q in TEST_QUERIES],
    "Type":               query_types,
    "Baseline_Precision": baseline_precision,
    "Enhanced_Precision": enhanced_precision,
    "Baseline_Recall":    baseline_recall,
    "Enhanced_Recall":    enhanced_recall,
    "Precision_Delta":    [e-b for e, b in
                           zip(enhanced_precision, baseline_precision)],
    "Recall_Delta":       [e-b for e, b in
                           zip(enhanced_recall, baseline_recall)],
})

results_df.to_csv("/content/evaluation_results.csv", index=False)
print("\n✓ Results saved to /content/evaluation_results.csv")

from google.colab import files
files.download("/content/evaluation_results.csv")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Evaluation model loaded.
Evaluating enhanced pipeline...
  ✓ What is the MSR activation threshold in the EU ETS?...
  ✓ What was the total EU ETS auction revenue in 2024?...
  ✓ What penalty do regulated entities pay for excess emiss...
  ✓ What percentage of global emissions does the EU ETS cov...
  ✓ How does auction volume reduction affect carbon price t...
  ✓ What is the relationship between free allocation and ca...
  ✓ How does CBAM interact with free allocation policy in t...
  ✓ What drove the decline in EU ETS emissions from power a...
  ✓ Is the carbon price high or low currently?...
  ✓ Should companies buy more allowances now?...
  ✓ What happens to the ETS if emissions drop significantly...
  ✓ How does the ETS affect industrial competitiveness?...
  ✓ What is the carbon price on Mars?...
  ✓ How does the EU ETS affect cryptocurrency mining?...
  ✓ What was the EUA price in 1990?...

Evaluating baseline pipeline...
  ✓ What is the MSR activation threshold in the EU ETS?

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>